# Fundación YPF - Data Science

## Entrega Final: Predicción de Tasas de Promoción Escolar en el Nivel Secundario mediante Regresión Lineal
**Alumnas:** Florencia Zoni, Virginia Contreras  
**Curso:** Data Science

---

## 1. Introducción y Contexto
En la gestión e investigación educativa, el seguimiento de las trayectorias escolares es fundamental para evaluar la estabilidad y el rendimiento del sistema educativo. Específicamente, la **tasa de promoción efectiva** mide el porcentaje de estudiantes que logran aprobar y pasar de año de manera regular.

Para este estudio, utilizamos la base de datos unificada `Tasa_de_Promocion_Efectiva_2012-2023_unificada.xlsx`, la cual recopila información estadística oficial estructurada por provincia y año (**unidad de análisis: provincia-año**) desde el año 2012 hasta el 2023.

### Objetivo
Predecir la tasa de promoción total de secundaria del año siguiente ($t+1$) a partir de las tasas registradas en el año actual ($t$) aplicando técnicas de Aprendizaje Automático Supervisado (**Regresión Lineal Múltiple**).

---

## 2. Estrategia Metodológica
1. **Carga y Limpieza de Datos (*Data Cleaning*):** Depuración de columnas que contienen valores nulos (como los años faltantes de primaria y secundaria inicial) para garantizar la consistencia del dataset.
2. **Creación de la Variable Objetivo (*Target*):** Generación de la columna $t+1$ (`Secundaria_Total_Siguiente`) mediante desplazamientos agrupados por jurisdicción.
3. **Definición de Características (*Features*):** Uso simultáneo de las tasas desglosadas por año escolar de secundaria (`Secundaria - Total`, `8° Año`, `9° Año`, `10° Año`, `11° Año` y `12° Año`) como variables predictoras multivariadas.
4. **Partición Temporal (*Train / Test Split*):** Entrenamiento del modelo utilizando los años estables de la serie histórica (**2012 a 2018**) y evaluación en los años posteriores (**2019 a 2022**), los cuales sufrieron el impacto de la pandemia de COVID-19.
5. **Evaluación de Métricas:** Medición del error absoluto y cuadrático medio ($MAE$, $RMSE$) y del coeficiente de determinación ($R^2$).

## 3. Desarrollo del Código y Modelado

### 3.1. Importación de Librerías y Carga de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Estilo de gráficos
sns.set_theme(style="whitegrid")

# Carga del archivo Excel
ruta_archivo = 'Tasa_de_Promocion_Efectiva_2012-2023_unificada.xlsx'
archivo_excel = pd.ExcelFile(ruta_archivo)
df = pd.read_excel(ruta_archivo, sheet_name=archivo_excel.sheet_names[0])
df.columns = df.columns.str.strip()

print("Columnas cargadas inicialmente:")
print(df.columns.tolist())

### 3.2. Limpieza de Datos y Preparación de la Serie Temporal

In [ ]:
# 1. Eliminar columnas con valores nulos
df_limpio_cols = df.dropna(axis=1, how='any').copy()

# 2. Ordenar cronológicamente por provincia y año
df_limpio_cols = df_limpio_cols.sort_values(by=['División político-territorial', 'Año']).reset_index(drop=True)

# 3. Crear la variable objetivo (Target) ANTES de filtrar nulos del target
df_limpio_cols['Secundaria_Total_Siguiente'] = df_limpio_cols.groupby('División político-territorial')['Secundaria - Total'].shift(-1)

# 4. Eliminar el último año de cada provincia (que no tiene año t+1)
df_modelo = df_limpio_cols.dropna(subset=['Secundaria_Total_Siguiente']).copy()

print(f"Dataset listo para modelar: {df_modelo.shape[0]} observaciones y {df_modelo.shape[1]} columnas.")

### 3.3. Análisis Exploratorio de Datos (EDA) - Visualización de la Serie Histórica

In [ ]:
# Gráfico para analizar la evolución temporal del promedio nacional
plt.figure(figsize=(10, 5))
promedio_anual = df_limpio_cols.groupby('Año')['Secundaria - Total'].mean()

plt.plot(promedio_anual.index, promedio_anual.values, marker='o', color='#2b5c8f', linewidth=2.5, label='Promedio Nacional')
plt.axvline(x=2018, color='#d9534f', linestyle='--', linewidth=2, label='Límite Train/Test (2018)')

plt.title('Evolución de la Tasa de Promoción Efectiva en Secundaria (2012 - 2023)', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Año', fontsize=11)
plt.ylabel('Tasa de Promoción (%)', fontsize=11)
plt.ylim(70, 100)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

### 3.4. Definición de Variables y Partición Train / Test

In [ ]:
# Definir características predictoras multivariadas
caracteristicas = [
    'Secundaria - Total',
    'Secundaria - 8° Año',
    'Secundaria - 9° Año',
    'Secundaria - 10° Año',
    'Secundaria - 11° Año',
    'Secundaria - 12° Año'
]

# Partición temporal estricta
anos_entrenamiento = [y for y in sorted(df_modelo['Año'].unique()) if y <= 2018]
anos_prueba = [y for y in sorted(df_modelo['Año'].unique()) if y > 2018]

df_entrenamiento = df_modelo[df_modelo['Año'].isin(anos_entrenamiento)]
df_prueba = df_modelo[df_modelo['Año'].isin(anos_prueba)]

X_train = df_entrenamiento[caracteristicas]
y_train = df_entrenamiento['Secundaria_Total_Siguiente']

X_test = df_prueba[caracteristicas]
y_test = df_prueba['Secundaria_Total_Siguiente']

print(f"Muestras de entrenamiento (2012-2018): {len(X_train)}")
print(f"Muestras de prueba (2019-2022): {len(X_test)}")

### 3.5. Entrenamiento del Modelo y Evaluación de Métricas

In [ ]:
# Ajustar el modelo de Regresión Lineal Múltiple
regresion_multiple = LinearRegression()
regresion_multiple.fit(X_train, y_train)

# Predicción en el conjunto de prueba
y_pred_multi = regresion_multiple.predict(X_test)

# Cálculo de métricas
rmse = np.sqrt(mean_squared_error(y_test, y_pred_multi))
mae = mean_absolute_error(y_test, y_pred_multi)
r2 = r2_score(y_test, y_pred_multi)

print("--- RESULTADOS DE LA REGRESIÓN MÚLTIPLE ---")
print(f"Intercepto: {regresion_multiple.intercept_:.4f}")
for feat, coef in zip(caracteristicas, regresion_multiple.coef_):
    print(f"Coeficiente ({feat}): {coef:.4f}")

print("\n--- MÉTRICAS EN TEST ---")
print(f"RMSE (Error Cuadrático Medio): {rmse:.4f} p.p.")
print(f"MAE (Error Absoluto Medio): {mae:.4f} p.p.")
print(f"R² (Coeficiente de Determinación): {r2:.4f}")

### 3.6. Gráfico de Diagnóstico: Valores Reales vs. Predichos

In [ ]:
# Gráfico de dispersión para comparar el comportamiento real vs predicho
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_multi, color='#4682b4', alpha=0.7, edgecolor='k', s=55, label='Provincias (Test: 2019-2022)')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='#e74c3c', linestyle='--', linewidth=2, label='Predicción Ideal (y = x)')

plt.title('Valores Reales vs. Valores Predichos (Tasa t+1)', fontsize=12, fontweight='bold')
plt.xlabel('Tasa Real de Promoción (%)', fontsize=10)
plt.ylabel('Tasa Predicha (%)', fontsize=10)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Conclusiones Finales

1. **Rendimiento General del Modelo:**
   El modelo entrenado con Regresión Lineal Múltiple obtuvo un **MAE de 4.76 p.p.** y un **RMSE de 6.56 p.p.** en el conjunto de prueba, lo que indica que sus predicciones se desviaron en promedio menos de 5 puntos porcentuales de la realidad.

2. **Impacto Estructural y *Data Drift* (Pandemia 2020):**
   El valor negativo de $R^2$ ($-0.5239$) en el período de prueba se explica directamente por el salto atípico en la tasa de promoción durante el año 2020 (que superó el 92% promedio a nivel nacional debido a las flexibilizaciones normativas del COVID-19). Como el modelo aprendió con datos estables pre-2018 (donde la tasa promediaba el 79%), tendió a **subestimar sistemáticamente** las tasas reales del período post-2019.

3. **Recomendación para Futuros Trabajos:**
   Para mejorar la capacidad de generalización del modelo frente a cambios estructurales, se sugiere explorar la inclusión de **efectos fijos provinciales** (variables *dummy* por jurisdicción) o incorporar variables macroeducativas y socioeconómicas adicionales.